# 02 Modeling & Feature Engineering
This phase will cover the feature engineering and modeling phase of the system before getting deployed.

Logistic Regression (Quit / Not Quitting), Base Random Forest, Tuned Random Forest using CV

- Since this project is a binary classification task (Determining whether an employee will leave or not), I will be running logistic regression, base random forest, and tuned random forest using GridSearchCV and conclude which performs the better and will be chosen as the champion model.



In [2]:
%pip install sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [ ]:
# Import packages
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay\
,classification_report, confusion_matrix, ConfusionMatrixDisplay
from statsmodels import api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression

import pickle

In [5]:
# Initialize file paths
processed_dataset_file_path = '../data/processed/'

df1 = pd.read_csv(processed_dataset_file_path + 'processed_dataset.csv')

In [6]:
df1.head(5)

,Unnamed: 0,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,work_accident,left,promotion_last_5years,department,salary
0,0,0.38,0.53,2,157,3,0,1,0,sales,low
1,1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,3,0.72,0.87,5,223,5,0,1,0,sales,low
4,4,0.37,0.52,2,159,3,0,1,0,sales,low


In [7]:
#Check Balance
df1['left'].value_counts(normalize=True)*100

left
0    83.39588
1    16.60412
Name: proportion, dtype: float64

In [9]:
# Create results helper function
def make_results(model_name:str,model_obj):
    '''This function accepts `model_name` (str) and `model_obj` (model object)
    as arguments and returns a pandas dataframe with the following metrics:
    f1, accuracy, recall, and precision
    '''
    cv_results = pd.DataFrame(model_obj.cv_results_)
    best_estimator_results_ = cv_results.iloc[cv_results['mean_test_f1'].idxmax()]
    f1 = best_estimator_results_.mean_test_f1
    accuracy = best_estimator_results_.mean_test_accuracy
    recall = best_estimator_results_.mean_test_recall
    precision = best_estimator_results_.mean_test_precision
    auc = best_estimator_results_.mean_test_roc_auc
    
    table = pd.DataFrame({
        'Model Name' : [model_name],
        'f1' : [f1],
        'accuracy' : [accuracy],
        'recall' : [recall],
        'precision' : [precision],
        'auc': [auc]
    })
    
    return table

In [10]:
# Create Table that will be used for storing model metrics
results_table = pd.DataFrame()

In [11]:
# Check for unique values and their counts for each categorical variables
df1[['department','salary']].value_counts()

department   salary
sales        low       1553
             medium    1449
technical    low       1138
             medium     940
support      low        867
             medium     828
IT           low        476
             medium     429
product_mng  low        343
RandD        medium     325
             low        322
marketing    low        310
             medium     301
accounting   low        296
hr           low        296
product_mng  medium     291
hr           medium     267
accounting   medium     262
sales        high       237
management   medium     169
technical    high       166
management   low        139
             high       128
support      high       126
IT           high        71
accounting   high        63
marketing    high        62
product_mng  high        52
RandD        high        47
hr           high        38
Name: count, dtype: int64

In [13]:
# Encode categorical variables
# For `salary` since it shows ordinality, I will be using a mapping function and One-Hot Encoding for department.
salary_map = {
    'high' : 2,
    'medium': 1,
    'low': 0
}

df2 = df1.copy()
df2['salary'] = df2['salary'].map(salary_map)

# Use get_dummmies to One-Hot encode `department`
df2 = pd.get_dummies(df2, columns=['department'], drop_first=True, dtype=int)
df2.head(10)

,Unnamed: 0,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,work_accident,left,promotion_last_5years,salary,department_RandD,department_accounting,department_hr,department_management,department_marketing,department_product_mng,department_sales,department_support,department_technical
0,0,0.38,0.53,2,157,3,0,1,0,0,0,0,0,0,0,0,1,0,0
1,1,0.80,0.86,5,262,6,0,1,0,1,0,0,0,0,0,0,1,0,0
2,2,0.11,0.88,7,272,4,0,1,0,1,0,0,0,0,0,0,1,0,0
3,3,0.72,0.87,5,223,5,0,1,0,0,0,0,0,0,0,0,1,0,0
4,4,0.37,0.52,2,159,3,0,1,0,0,0,0,0,0,0,0,1,0,0
5,5,0.41,0.50,2,153,3,0,1,0,0,0,0,0,0,0,0,1,0,0
6,6,0.10,0.77,6,247,4,0,1,0,0,0,0,0,0,0,0,1,0,0
7,7,0.92,0.85,5,259,5,0,1,0,0,0,0,0,0,0,0,1,0,0
8,8,0.89,1.00,5,224,5,0,1,0,0,0,0,0,0,0,0,1,0,0
9,9,0.42,0.53,2,142,3,0,1,0,0,0,0,0,0,0,0,1,0,0
